In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.ndimage import gaussian_filter1d

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'

palette = ['#C0D6CA','#78ACA8','#2D6B8F','#235796',
           '#E7C4C0','#E3A39A','#CA6F6A','#7B3841',
           '#D5BC67','#20425B','#E77A5B','#9C9DB2']

col_fh       = '#E3A39A'
col_ar       = palette[7]
alphas       = [0.5, 0.75, 1.0]
patch_alphas = [0.10, 0.2, 0.4]   # all visible, all low enough for grid to show

T_context = 50
patch_size = 20
n_patches  = 3
T_future   = patch_size * n_patches
t_all      = np.arange(T_context + T_future)

y_all = (np.sin(0.28 * t_all) + 0.4 * np.sin(0.11 * t_all + 1.2)
         + 0.012 * t_all + np.random.normal(0, 0.22, len(t_all)))
y_context = y_all[:T_context]
y_future  = y_all[T_context:]
t_future  = np.arange(T_context, T_context + T_future)

fh_pred = y_future + np.random.normal(0, 0.45, T_future)
fh_pred = gaussian_filter1d(fh_pred, sigma=0.8)

ar_pred = np.zeros(T_future)
prev = y_context[-1]
drift = 0.0
for i in range(T_future):
    patch_idx  = i // patch_size
    drift_rate = [0.04, 0.09, 0.20][patch_idx]
    noise_rate = [0.35, 0.50, 0.70][patch_idx]
    true_step  = (np.sin(0.28*(T_context+i)) + 0.4*np.sin(0.11*(T_context+i)+1.2)
                  - np.sin(0.28*(T_context+i-1)) - 0.4*np.sin(0.11*(T_context+i-1)+1.2)
                  + 0.012)
    drift += np.random.normal(0, drift_rate)
    pred   = prev + true_step + np.random.normal(0, noise_rate) + drift * 0.35
    ar_pred[i] = pred
    prev = pred
ar_pred = gaussian_filter1d(ar_pred, sigma=0.6)

HOLD     = 40
N_STAGES = 5
TOTAL    = N_STAGES * HOLD

fig, ax = plt.subplots(figsize=(15, 4))
ylim_top = 3.2

ax.axvline(T_context, color='black', lw=1.4, linestyle='-', alpha=0.55)
ax.text(T_context + 0.4, -1.8, 'Forecast Start', fontsize=16, color='#555', va='top')

for i in range(n_patches):
    x0 = T_context + i * patch_size
    ax.axvspan(x0, x0 + patch_size,
               alpha=patch_alphas[i], color=col_ar, zorder=0)
    ax.axvline(x0, color='#888780', lw=0.9, linestyle=':', alpha=alphas[i])
    ax.text(x0 + patch_size / 2, ylim_top + 3,
            f'Patch {i+1}', fontsize=16, ha='center',
            color=col_ar, alpha=alphas[i], fontweight='bold')

ax.set_xlim(-2, T_context + T_future + 9)
ax.set_ylim(-2.5, ylim_top + 3)
ax.set_xlabel('Time step', fontsize=16, labelpad=12)
ax.set_ylabel('Value', fontsize=16)
ax.tick_params(axis='x', labelsize=16, colors='black')
ax.tick_params(axis='y', labelsize=16, colors='black')
ax.spines[['top','right']].set_visible(False)
ax.grid(True, color='#d0d0d0', linewidth=0.7, alpha=0.9, zorder=1)
ax.set_facecolor('white')
fig.patch.set_facecolor('white')

line_context, = ax.plot([], [], color='black', lw=2, label='Ground truth', zorder=3)
line_gt_fut,  = ax.plot([], [], color='black', lw=2, linestyle='--', zorder=3)
line_fh,      = ax.plot([], [], color=col_fh, lw=2.2, label='Fixed-horizon', zorder=3)
line_ar = []
for i in range(n_patches):
    lbl = 'Autoregressive' if i == 0 else None
    l, = ax.plot([], [], color=col_ar, lw=2.2, alpha=alphas[i], label=lbl, zorder=3)
    line_ar.append(l)

arrow_obj = [None]
ax.legend(fontsize=16, loc='upper left', frameon=True)

def animate(frame):
    stage = frame // HOLD

    if stage >= 0:
        line_context.set_data(np.arange(T_context), y_context)

    if stage >= 1:
        line_fh.set_data(t_future, fh_pred)
        line_gt_fut.set_data(t_future, y_future)
        i0, i1 = 0, min(patch_size + 1, T_future)
        line_ar[0].set_data(t_future[i0:i1], ar_pred[i0:i1])

    if stage >= 2:
        i0, i1 = patch_size, min(2 * patch_size + 1, T_future)
        line_ar[1].set_data(t_future[i0:i1], ar_pred[i0:i1])

    if stage >= 3:
        i0, i1 = 2 * patch_size, T_future
        line_ar[2].set_data(t_future[i0:i1], ar_pred[i0:i1])

    if stage >= 4 and arrow_obj[0] is None:
        arrow_obj[0] = ax.annotate(
            '', xy=(T_context + T_future - 1, ar_pred[-1]),
            xytext=(T_context + T_future - 1, fh_pred[-1]),
            arrowprops=dict(arrowstyle='<->', color=col_ar, lw=1.6))

    return [line_context, line_gt_fut, line_fh] + line_ar

ani = animation.FuncAnimation(fig, animate, frames=TOTAL, interval=40, blit=False)
plt.tight_layout()
ani.save('exposure_bias.gif', writer='pillow', fps=25,
         savefig_kwargs={'facecolor': 'white'})
print("done")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.ndimage import gaussian_filter1d

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'

palette = ['#C0D6CA','#78ACA8','#2D6B8F','#235796',
           '#E7C4C0','#E3A39A','#CA6F6A','#7B3841',
           '#D5BC67','#20425B','#E77A5B','#9C9DB2']

col_ar   = palette[7]
col_q    = palette[1]
alphas       = [0.5, 0.75, 1.0]
patch_alphas = [0.10, 0.20, 0.40]

T_context  = 50
patch_size = 20
n_patches  = 3
T_future   = patch_size * n_patches
t_all      = np.arange(T_context + T_future)

y_all = (np.sin(0.28 * t_all) + 0.4 * np.sin(0.11 * t_all + 1.2)
         + 0.012 * t_all + np.random.normal(0, 0.22, len(t_all)))
y_context = y_all[:T_context]
y_future  = y_all[T_context:]
t_future  = np.arange(T_context, T_context + T_future)

# Advance RNG to match exposure_bias.py
_ = gaussian_filter1d(y_future + np.random.normal(0, 0.45, T_future), sigma=0.8)

# AR median — identical to exposure_bias.py
ar_median = np.zeros(T_future)
prev = y_context[-1]
drift = 0.0
for i in range(T_future):
    patch_idx  = i // patch_size
    drift_rate = [0.04, 0.09, 0.20][patch_idx]
    noise_rate = [0.35, 0.50, 0.70][patch_idx]
    true_step  = (np.sin(0.28*(T_context+i)) + 0.4*np.sin(0.11*(T_context+i)+1.2)
                  - np.sin(0.28*(T_context+i-1)) - 0.4*np.sin(0.11*(T_context+i-1)+1.2)
                  + 0.012)
    drift += np.random.normal(0, drift_rate)
    pred   = prev + true_step + np.random.normal(0, noise_rate) + drift * 0.35
    ar_median[i] = pred
    prev = pred
ar_median = gaussian_filter1d(ar_median, sigma=0.6)

# Option A: quantiles computed by conditioning only on median at each step
# Uncertainty is FIXED — one-step-ahead spread only, doesn't compound
# This simulates: model always sees Q50 as input, so spread is constant
one_step_spread = np.array([0.35, 0.50, 0.70])  # per patch noise_rate

q_offsets = {'q10': -1.28, 'q25': -0.67, 'q50': 0.0, 'q75': 0.67, 'q90': 1.28}

# Fixed spread per patch — does NOT grow because we condition on median only
spreads = np.zeros(T_future)
for i in range(T_future):
    spreads[i] = one_step_spread[i // patch_size]

q_data = {}
for q, z in q_offsets.items():
    q_data[q] = ar_median + z * spreads   # parallel bands, constant width per patch

HOLD     = 45
N_STAGES = 5
TOTAL    = N_STAGES * HOLD

fig, ax = plt.subplots(figsize=(15, 4))
ylim_top = 3.2

ax.axvline(T_context, color='black', lw=1.4, linestyle='-', alpha=0.55)
ax.text(T_context + 0.4, -3.0, 'Forecast Start', fontsize=16, color='#555', va='top')

for i in range(n_patches):
    x0 = T_context + i * patch_size
    ax.axvspan(x0, x0 + patch_size, alpha=patch_alphas[i], color=col_ar, zorder=0)
    ax.axvline(x0, color='#888780', lw=0.9, linestyle=':', alpha=alphas[i])
    ax.text(x0 + patch_size / 2, ylim_top + 3,
            f'Patch {i+1}', fontsize=16, ha='center',
            color=col_ar, alpha=alphas[i], fontweight='bold')

ax.set_xlim(-2, T_context + T_future + 10)
ax.set_ylim(-4.2, ylim_top + 3)
ax.set_xlabel('Time step', fontsize=16, labelpad=12)
ax.set_ylabel('Value', fontsize=16)
ax.tick_params(axis='x', labelsize=16, colors='black')
ax.tick_params(axis='y', labelsize=16, colors='black')
ax.spines[['top','right']].set_visible(False)
ax.grid(True, color='#d0d0d0', linewidth=0.7, alpha=0.9, zorder=1)
ax.set_facecolor('white')
fig.patch.set_facecolor('white')

line_context, = ax.plot([], [], color='black', lw=2,   label='Ground truth', zorder=5)
line_gt_fut,  = ax.plot([], [], color='black', lw=2,   linestyle='--', zorder=5)
line_median,  = ax.plot([], [], color=col_ar,  lw=2.2, label='AR median (conditioning input)', zorder=6)

# Quantile bands per patch
band80, band50 = [], []
for i in range(n_patches):
    i0, i1 = i * patch_size, (i+1) * patch_size
    f80 = ax.fill_between(t_future[i0:i1], q_data['q10'][i0:i1], q_data['q90'][i0:i1],
                           alpha=0, color=col_q, zorder=3)
    f50 = ax.fill_between(t_future[i0:i1], q_data['q25'][i0:i1], q_data['q75'][i0:i1],
                           alpha=0, color=col_q, zorder=3)
    band80.append(f80)
    band50.append(f50)

# Quantile lines per patch
q_styles = {
    'q10': dict(color=col_q, lw=1.2, linestyle='--', alpha=0.65, zorder=4),
    'q25': dict(color=col_q, lw=1.4, linestyle='--', alpha=0.80, zorder=4),
    'q50': dict(color=col_q, lw=2.0, linestyle='-',  alpha=1.00, zorder=4),
    'q75': dict(color=col_q, lw=1.4, linestyle='--', alpha=0.80, zorder=4),
    'q90': dict(color=col_q, lw=1.2, linestyle='--', alpha=0.65, zorder=4),
}
q_lines = {q: [] for q in q_data}
for q in q_lines:
    for i in range(n_patches):
        l, = ax.plot([], [], **q_styles[q])
        q_lines[q].append(l)

# Annotation text
note_text = [None]

ax.legend(fontsize=13, loc='upper left', frameon=True,
          handles=[
              plt.Line2D([0],[0], color='black', lw=2),
              plt.Line2D([0],[0], color=col_ar,  lw=2.2),
              plt.Line2D([0],[0], color=col_q,   lw=2.0),
              plt.Line2D([0],[0], color=col_q,   lw=1.2, linestyle='--', alpha=0.7),
          ],
          labels=['Ground truth', 'AR median (fed back each step)',
                  'Q50', 'Q10 / Q25 / Q75 / Q90'])

def show_patch(p):
    i0, i1 = p * patch_size, min((p+1) * patch_size, T_future)
    for q in q_lines:
        q_lines[q][p].set_data(t_future[i0:i1], q_data[q][i0:i1])
    band80[p].set_alpha(0.15)
    band50[p].set_alpha(0.28)

def animate(frame):
    stage = frame // HOLD

    if stage >= 0:
        line_context.set_data(np.arange(T_context), y_context)
        line_gt_fut.set_data(t_future, y_future)

    if stage >= 1:
        line_median.set_data(t_future[:patch_size], ar_median[:patch_size])
        show_patch(0)

    if stage >= 2:
        line_median.set_data(t_future[:2*patch_size], ar_median[:2*patch_size])
        show_patch(1)

    if stage >= 3:
        line_median.set_data(t_future, ar_median)
        show_patch(2)

    return [line_context, line_gt_fut, line_median]

ani = animation.FuncAnimation(fig, animate, frames=TOTAL, interval=40, blit=False)
plt.tight_layout()
ani.save('median_conditioning.gif', writer='pillow', fps=25,
         savefig_kwargs={'facecolor': 'white'})
print("done")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.ndimage import gaussian_filter1d

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'

palette = ['#C0D6CA','#78ACA8','#2D6B8F','#235796',
           '#E7C4C0','#E3A39A','#CA6F6A','#7B3841',
           '#D5BC67','#20425B','#E77A5B','#9C9DB2']

col_ar   = palette[7]
col_path = palette[2]
col_q    = palette[1]
alphas       = [0.5, 0.75, 1.0]
patch_alphas = [0.10, 0.20, 0.40]

T_context  = 50
patch_size = 20
n_patches  = 3
T_future   = patch_size * n_patches
N_SAMPLES  = 200
t_all      = np.arange(T_context + T_future)

y_all = (np.sin(0.28 * t_all) + 0.4 * np.sin(0.11 * t_all + 1.2)
         + 0.012 * t_all + np.random.normal(0, 0.22, len(t_all)))
y_context = y_all[:T_context]
y_future  = y_all[T_context:]
t_future  = np.arange(T_context, T_context + T_future)

# Advance RNG identically to exposure_bias.py (fh_pred consumed these numbers)
_ = gaussian_filter1d(y_future + np.random.normal(0, 0.45, T_future), sigma=0.8)

# AR median — identical code to ar_pred in exposure_bias.py
ar_median = np.zeros(T_future)
prev = y_context[-1]
drift = 0.0
for i in range(T_future):
    patch_idx  = i // patch_size
    drift_rate = [0.04, 0.09, 0.20][patch_idx]
    noise_rate = [0.35, 0.50, 0.70][patch_idx]
    true_step  = (np.sin(0.28*(T_context+i)) + 0.4*np.sin(0.11*(T_context+i)+1.2)
                  - np.sin(0.28*(T_context+i-1)) - 0.4*np.sin(0.11*(T_context+i-1)+1.2)
                  + 0.012)
    drift += np.random.normal(0, drift_rate)
    pred   = prev + true_step + np.random.normal(0, noise_rate) + drift * 0.35
    ar_median[i] = pred
    prev = pred
ar_median = gaussian_filter1d(ar_median, sigma=0.6)

# Sample paths: fan out from ar_median with compounding uncertainty
np.random.seed(99)
noise_scales = [0.30, 0.55, 0.90]
paths = np.zeros((N_SAMPLES, T_future))
for s in range(N_SAMPLES):
    cumulative_offset = 0.0
    for i in range(T_future):
        patch_idx = i // patch_size
        cumulative_offset += np.random.normal(0, noise_scales[patch_idx] * 0.3)
        paths[s, i] = ar_median[i] + cumulative_offset + np.random.normal(0, noise_scales[patch_idx] * 0.4)

q10 = np.percentile(paths, 10, axis=0)
q25 = np.percentile(paths, 25, axis=0)
q50 = np.percentile(paths, 50, axis=0)
q75 = np.percentile(paths, 75, axis=0)
q90 = np.percentile(paths, 90, axis=0)

HOLD     = 45
N_STAGES = 5
TOTAL    = N_STAGES * HOLD

fig, ax = plt.subplots(figsize=(15, 4))
ylim_top = 3.2

ax.axvline(T_context, color='black', lw=1.4, linestyle='-', alpha=0.55)
ax.text(T_context + 0.4, -3.2, 'Forecast Start', fontsize=16, color='#555', va='top')

for i in range(n_patches):
    x0 = T_context + i * patch_size
    ax.axvspan(x0, x0 + patch_size, alpha=patch_alphas[i], color=col_ar, zorder=0)
    ax.axvline(x0, color='#888780', lw=0.9, linestyle=':', alpha=alphas[i])
    ax.text(x0 + patch_size / 2, ylim_top + 3,
            f'Patch {i+1}', fontsize=16, ha='center',
            color=col_ar, alpha=alphas[i], fontweight='bold')

ax.set_xlim(-2, T_context + T_future + 10)
ax.set_ylim(-4.0, ylim_top + 3)
ax.set_xlabel('Time step', fontsize=16, labelpad=12)
ax.set_ylabel('Value', fontsize=16)
ax.tick_params(axis='x', labelsize=16, colors='black')
ax.tick_params(axis='y', labelsize=16, colors='black')
ax.spines[['top','right']].set_visible(False)
ax.grid(True, color='#d0d0d0', linewidth=0.7, alpha=0.9, zorder=1)
ax.set_facecolor('white')
fig.patch.set_facecolor('white')

line_context, = ax.plot([], [], color='black', lw=2,   label='Ground truth', zorder=5)
line_gt_fut,  = ax.plot([], [], color='black', lw=2,   linestyle='--', zorder=5)
line_median,  = ax.plot([], [], color=col_ar,  lw=2.2, label='AR median', zorder=6)

path_lines = []
for s in range(N_SAMPLES):
    l, = ax.plot([], [], color=col_path, lw=0.4, alpha=0.10, zorder=2)
    path_lines.append(l)

band80, band50 = [], []
for i in range(n_patches):
    i0, i1 = i * patch_size, (i+1) * patch_size
    f80 = ax.fill_between(t_future[i0:i1], q10[i0:i1], q90[i0:i1],
                           alpha=0, color=col_q, zorder=3)
    f50 = ax.fill_between(t_future[i0:i1], q25[i0:i1], q75[i0:i1],
                           alpha=0, color=col_q, zorder=3)
    band80.append(f80)
    band50.append(f50)

q_data   = {'q10': q10, 'q25': q25, 'q50': q50, 'q75': q75, 'q90': q90}
q_styles = {
    'q10': dict(color=col_q, lw=1.2, linestyle='--', alpha=0.65, zorder=4),
    'q25': dict(color=col_q, lw=1.4, linestyle='--', alpha=0.80, zorder=4),
    'q50': dict(color=col_q, lw=2.0, linestyle='-',  alpha=1.00, zorder=4),
    'q75': dict(color=col_q, lw=1.4, linestyle='--', alpha=0.80, zorder=4),
    'q90': dict(color=col_q, lw=1.2, linestyle='--', alpha=0.65, zorder=4),
}
q_lines = {q: [] for q in q_data}
for q in q_lines:
    for i in range(n_patches):
        l, = ax.plot([], [], **q_styles[q])
        q_lines[q].append(l)

arrow_obj = [None]
unc_text  = [None]

ax.legend(fontsize=13, loc='upper left', frameon=True,
          handles=[
              plt.Line2D([0],[0], color='black', lw=2),
              plt.Line2D([0],[0], color=col_ar,   lw=2.2),
              plt.Line2D([0],[0], color=col_path, lw=1.5, alpha=0.5),
              plt.Line2D([0],[0], color=col_q,    lw=2.0),
              plt.Line2D([0],[0], color=col_q,    lw=1.2, linestyle='--', alpha=0.7),
          ],
          labels=['Ground truth', 'AR median', 'Sample paths',
                  'Q50', 'Q10 / Q25 / Q75 / Q90'])

def show_patch(p):
    i0, i1 = p * patch_size, min((p+1) * patch_size, T_future)
    for s in range(N_SAMPLES):
        path_lines[s].set_data(t_future[:i1], paths[s, :i1])
    for q in q_lines:
        q_lines[q][p].set_data(t_future[i0:i1], q_data[q][i0:i1])
    band80[p].set_alpha(0.15)
    band50[p].set_alpha(0.28)

def animate(frame):
    stage = frame // HOLD

    if stage >= 0:
        line_context.set_data(np.arange(T_context), y_context)
        line_gt_fut.set_data(t_future, y_future)

    if stage >= 1:
        line_median.set_data(t_future[:patch_size], ar_median[:patch_size])
        show_patch(0)

    if stage >= 2:
        line_median.set_data(t_future[:2*patch_size], ar_median[:2*patch_size])
        show_patch(1)

    if stage >= 3:
        line_median.set_data(t_future, ar_median)
        show_patch(2)

    return [line_context, line_gt_fut, line_median] + path_lines

ani = animation.FuncAnimation(fig, animate, frames=TOTAL, interval=40, blit=False)
plt.tight_layout()
ani.save('joint_distribution.gif', writer='pillow', fps=25,
         savefig_kwargs={'facecolor': 'white'})
print("done")